# UniVerse Creator Engine
Runtime de IA do UniVerse Creator. Não altera a interface do aplicativo.

Conecta Google Drive, ComfyUI, Stable Diffusion XL, ControlNet, AnimateDiff, Kohya_ss e Stable Video Diffusion quando os pesos/licenças estiverem disponíveis.

Selecione GPU no Colab e execute as células em ordem. Para modelos com licença aceita, o Hugging Face pode exigir HF_TOKEN.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os
ROOT=Path('/content/drive/MyDrive/UniVerseCreator')
for name in ['identities','generated','loras','models/checkpoints','models/controlnet','models/animatediff','models/svd']:
    (ROOT/name).mkdir(parents=True,exist_ok=True)
print('Google Drive conectado:', ROOT)


In [ ]:
import os, subprocess, sys
!git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt fastapi uvicorn requests python-multipart huggingface_hub hf_transfer
!mkdir -p custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git custom_nodes/ComfyUI-Manager
!git clone https://github.com/Fannovel16/comfyui_controlnet_aux.git custom_nodes/comfyui_controlnet_aux
!git clone https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git custom_nodes/ComfyUI-AnimateDiff-Evolved
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git custom_nodes/ComfyUI-VideoHelperSuite
!for f in custom_nodes/comfyui_controlnet_aux/requirements.txt custom_nodes/ComfyUI-AnimateDiff-Evolved/requirements.txt custom_nodes/ComfyUI-VideoHelperSuite/requirements.txt; do [ ! -f $f ] || pip install -q -r $f; done
!git clone https://github.com/bmaltais/kohya_ss.git /content/kohya_ss
!cd /content/kohya_ss && pip install -q -r requirements.txt
!rm -rf /content/ComfyUI/models/checkpoints /content/ComfyUI/models/controlnet /content/ComfyUI/models/loras /content/ComfyUI/models/animatediff
!ln -s /content/drive/MyDrive/UniVerseCreator/models/checkpoints /content/ComfyUI/models/checkpoints
!ln -s /content/drive/MyDrive/UniVerseCreator/models/controlnet /content/ComfyUI/models/controlnet
!ln -s /content/drive/MyDrive/UniVerseCreator/loras /content/ComfyUI/models/loras
!mkdir -p /content/ComfyUI/models/animatediff
!ln -s /content/drive/MyDrive/UniVerseCreator/models/animatediff /content/ComfyUI/models/animatediff
print('Bibliotecas e nós instalados. Modelos serão baixados na próxima célula.')


In [ ]:
from huggingface_hub import hf_hub_download
import os, shutil
HF_TOKEN=os.environ.get('HF_TOKEN') or None
def fetch_model(repo,filename,dest,token=HF_TOKEN):
    target=Path(dest); target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists() and target.stat().st_size>100_000_000:
        print('Já existe:',target.name); return True
    try:
        downloaded=hf_hub_download(repo_id=repo,filename=filename,token=token)
        shutil.copy2(downloaded,target); print('Modelo instalado:',target); return True
    except Exception as e:
        print('Não foi possível baixar',repo,filename,'|',str(e)[:350]); return False
# SDXL Base 1.0. O repositório pode exigir aceite prévio dos termos da Stability AI.
sdxl_ok=fetch_model('stabilityai/stable-diffusion-xl-base-1.0','sd_xl_base_1.0.safetensors',ROOT/'models/checkpoints/sd_xl_base_1.0.safetensors')
# Stable Diffusion 1.5 para o pipeline AnimateDiff.
sd15_ok=fetch_model('stable-diffusion-v1-5/stable-diffusion-v1-5','v1-5-pruned-emaonly.safetensors',ROOT/'models/checkpoints/v1-5-pruned-emaonly.safetensors')
# ControlNet OpenPose SDXL. Arquivo do modelo oficial do repositório Xinsir.
controlnet_ok=fetch_model('xinsir/controlnet-openpose-sdxl-1.0','diffusion_pytorch_model.safetensors',ROOT/'models/controlnet/controlnet-openpose-sdxl.safetensors')
# Motion module AnimateDiff SD1.5. É instalado separadamente do checkpoint SDXL.
animatediff_ok=fetch_model('guoyww/animatediff','mm_sd_v15_v2.ckpt',ROOT/'models/animatediff/mm_sd_v15_v2.ckpt')
# SVD é distribuído com acesso sujeito aos termos da Stability AI. Se autorizado, baixe os pesos no repositório oficial.
svd_ok=fetch_model('stabilityai/stable-video-diffusion-img2vid','svd.safetensors',ROOT/'models/checkpoints/svd.safetensors')
print({'SDXL':sdxl_ok,'SD1.5':sd15_ok,'ControlNet':controlnet_ok,'AnimateDiff':animatediff_ok,'SVD':svd_ok})


In [ ]:
import os, subprocess, threading, time, requests
from google.colab import output
os.environ['COMFYUI_URL']='http://127.0.0.1:8188'
os.environ['COMFYUI_CHECKPOINT']='sd_xl_base_1.0.safetensors'
os.environ['COMFYUI_SD15_CHECKPOINT']='v1-5-pruned-emaonly.safetensors'
if (ROOT/'models/controlnet/controlnet-openpose-sdxl.safetensors').exists(): os.environ['COMFYUI_CONTROLNET']='controlnet-openpose-sdxl.safetensors'
if (ROOT/'models/animatediff/mm_sd_v15_v2.ckpt').exists(): os.environ['COMFYUI_MOTION_MODULE']='mm_sd_v15_v2.ckpt'
# AnimateDiff-Evolved procura motion modules nesta pasta padrão.
!mkdir -p /content/ComfyUI/models/animatediff_models
!ln -sfn /content/drive/MyDrive/UniVerseCreator/models/animatediff /content/ComfyUI/models/animatediff_models
if (ROOT/'models/checkpoints/svd.safetensors').exists(): os.environ['COMFYUI_SVD_CHECKPOINT']='svd.safetensors'
comfy=subprocess.Popen(['python','main.py','--listen','127.0.0.1','--port','8188'],cwd='/content/ComfyUI',stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get('http://127.0.0.1:8188/system_stats',timeout=3).raise_for_status(); break
    except Exception: time.sleep(2)
else: raise RuntimeError('ComfyUI não iniciou. Confira a saída da instalação.')
%cd /content
!wget -q -O /content/uvc_engine.py https://raw.githubusercontent.com/melpfintner/universe-creator/main/colab/uvc_engine.py
def run_api():
    import uvicorn
    uvicorn.run('uvc_engine:app',host='0.0.0.0',port=8000,log_level='info')
threading.Thread(target=run_api,daemon=True).start()
time.sleep(3)
public_url=output.eval_js('google.colab.kernel.proxyPort(8000)')
print('URL HTTPS DO COLAB:',public_url)
print('COLAB_MODULE1_URL =',public_url.rstrip('/')+'/create_identity')
print('COLAB_MODULE2_URL =',public_url.rstrip('/')+'/generate_image')
print('API health:',requests.get(public_url.rstrip('/')+'/health').json())


In [ ]:
## Recursos e limitações
- O SDXL e o ControlNet são baixados automaticamente quando o repositório permite acesso. Modelos gated exigem aceite dos termos na Hugging Face e, se solicitado, HF_TOKEN.
- O endpoint `/generate_image` executa ComfyUI e aceita ControlNet por meio de `controlImage` e `controlnet: true`. ControlNet controla estrutura/pose, não garante identidade facial.
- O endpoint `/identity/{identity_id}/dataset` recebe fotos para treinamento; `/train_lora` inicia o treinamento Kohya_ss com pelo menos cinco imagens; `/train_lora/{identity_id}/status` informa andamento.
- AnimateDiff-Evolved e VideoHelperSuite ficam instalados no ComfyUI. SVD usa workflow dedicado quando o checkpoint estiver disponível. Pesos de vídeo podem exigir aceite separado de licença.
- Para consistência facial real, é necessário treinar LoRA com fotos suficientes. Uma única foto de referência não equivale a um LoRA.
- A URL pública do Colab muda quando o runtime reinicia. As variáveis COLAB_MODULE1_URL e COLAB_MODULE2_URL do Worker precisam apontar para a URL exibida.


## Observações técnicas
- A geração real exige um checkpoint SDXL no Google Drive. Sem pesos, o ComfyUI não consegue gerar imagens.
- ControlNet precisa dos modelos de controle correspondentes e dos nós auxiliares instalados no ComfyUI. A rota base não finge que ControlNet ou AnimateDiff estão ativos.
- Kohya_ss exige um conjunto de fotos da identidade, captions e GPU adequada. Não inicie treinamento com apenas uma foto.
- AnimateDiff exige motion module compatível e workflow ComfyUI específico; a rota de vídeo só deve ser habilitada após instalar e validar esses componentes.
- O endereço público do Colab muda quando o runtime reinicia. Atualize as duas variáveis do Worker quando isso ocorrer.